# 12: Torus Surgery Workflow (Julia-Accelerated)

We compare:
1. a normal torus, and
2. a torus with an extra handle (genus increased).

Then we:
- triangulate both,
- visualize triangulations and homology generators (`H0`, `H1`, `H2`),
- run pySurgery homeomorphism diagnostics,
- perform a handle-removal surgery step,
- verify that topology matches the normal torus after surgery.

## Environment Check (Optional one-time installs)

If needed in your active kernel environment:
- `pip install juliacall plotly`
- Julia package once: `import Pkg; Pkg.add("AbstractAlgebra")`

In [4]:
try:
    from juliacall import Main as jl
    jl.seval('import Pkg; Pkg.add("AbstractAlgebra")')
    jl.seval('using AbstractAlgebra')
    print("Julia package warm-up complete.")
except Exception as exc:
    print("Julia warm-up skipped/failed:", repr(exc))

   Resolving package versions...
  No Changes to `~/Desktop/SurgeryTheory/.venv/julia_env/Project.toml`
  No Changes to `~/Desktop/SurgeryTheory/.venv/julia_env/Manifest.toml`


Julia package warm-up complete.


In [5]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from pysurgery.bridge.julia_bridge import julia_engine
from pysurgery.integrations.gudhi_bridge import triangulate_surface, extract_complex_data
from pysurgery.core.complexes import ChainComplex
from pysurgery.core.homology_generators import (
    compute_homology_basis_from_simplex_tree,
    compute_optimal_h1_basis_from_simplex_tree,
)
from pysurgery.homeomorphism import analyze_homeomorphism_2d_result
from pysurgery.homeomorphism_witness import build_homeomorphism_witness

print("Julia available:", julia_engine.available)
if not julia_engine.available:
    raise RuntimeError(
        "Julia backend is not available for this notebook run. "
        f"Bridge error: {julia_engine.error}"
    )

Julia available: True


## Geometry Builders
- `sample_torus`: genus-1 reference surface.
- `sample_double_torus_with_bridge`: genus-2 style surface (torus + extra handle).

In [6]:
def sample_torus(nu=120, nv=60, R=3.0, r=1.0, center=(0.0, 0.0, 0.0), noise=0.0, seed=0):
    rng = np.random.default_rng(seed)
    u = np.linspace(0.0, 2.0 * np.pi, nu, endpoint=False)
    v = np.linspace(0.0, 2.0 * np.pi, nv, endpoint=False)
    U, V = np.meshgrid(u, v, indexing="xy")

    x = (R + r * np.cos(V)) * np.cos(U)
    y = (R + r * np.cos(V)) * np.sin(U)
    z = r * np.sin(V)

    pts = np.column_stack([x.ravel(), y.ravel(), z.ravel()])
    pts += np.array(center, dtype=float)

    if noise > 0.0:
        pts += noise * rng.normal(size=pts.shape)
    return pts


def sample_double_torus_with_bridge(
    nu=90,
    nv=45,
    R=2.2,
    r=0.75,
    offset=3.0,
    bridge_radius=0.32,
    nt=120,
    ntheta=36,
    noise=0.0,
    seed=1,
):
    rng = np.random.default_rng(seed)

    left = sample_torus(nu=nu, nv=nv, R=R, r=r, center=(-offset, 0.0, 0.0), noise=0.0, seed=seed)
    right = sample_torus(nu=nu, nv=nv, R=R, r=r, center=(+offset, 0.0, 0.0), noise=0.0, seed=seed + 11)

    # Bridge tube along x-axis between the two tori (adds an extra handle)
    t = np.linspace(-offset + R, offset - R, nt, endpoint=True)
    th = np.linspace(0.0, 2.0 * np.pi, ntheta, endpoint=False)
    T, TH = np.meshgrid(t, th, indexing="xy")

    bx = T
    by = bridge_radius * np.cos(TH)
    bz = bridge_radius * np.sin(TH)
    bridge = np.column_stack([bx.ravel(), by.ravel(), bz.ravel()])

    pts = np.vstack([left, right, bridge])
    if noise > 0.0:
        pts += noise * rng.normal(size=pts.shape)
    return pts

## Build Point Clouds (size chosen so triangulation can use Julia path)
`triangulate_surface` switches to Julia triangulation when point count is large (`>5000` in current API).

In [7]:
clean_points = sample_torus(
    nu=120, nv=60, R=3.0, r=1.0, center=(0.0, 0.0, 0.0), noise=0.005, seed=7
)

handled_points = sample_double_torus_with_bridge(
    nu=80, nv=45, R=2.2, r=0.75, offset=3.0, bridge_radius=0.30,
    nt=140, ntheta=38, noise=0.005, seed=9
)

print("clean_points shape:", clean_points.shape)
print("handled_points shape:", handled_points.shape)

clean_points shape: (7200, 3)
handled_points shape: (12520, 3)


## Topology Extraction Utilities

In [ ]:
def simplex_counts(st):
    skel = list(st.get_skeleton(2))
    V = st.num_vertices()
    E = sum(1 for s, _ in skel if len(s) == 2)
    F = sum(1 for s, _ in skel if len(s) == 3)
    return V, E, F, (V - E + F)


def build_surface_record(points, name):
    st = triangulate_surface(points)  # Julia-accelerated path for large clouds
    boundaries, cells, _, _ = extract_complex_data(st)  # Uses Julia when available
    cc = ChainComplex(boundaries=boundaries, dimensions=sorted(cells.keys()), coefficient_ring="Z")

    h0 = cc.homology(0)
    h1 = cc.homology(1)
    h2 = cc.homology(2)

    g0 = compute_homology_basis_from_simplex_tree(st, dimension=0)
    g1 = compute_optimal_h1_basis_from_simplex_tree(st, point_cloud=points, max_cycles=8)
    g2 = compute_homology_basis_from_simplex_tree(st, dimension=2, mode="valid")

    V, E, F, chi = simplex_counts(st)

    return {
        "name": name,
        "points": points,
        "st": st,
        "cc": cc,
        "cells": cells,
        "counts": {"V": V, "E": E, "F": F, "chi": chi},
        "homology": {"H0": h0, "H1": h1, "H2": h2},
        "generators": {"H0": g0, "H1": g1, "H2": g2},
    }


clean = build_surface_record(clean_points, "clean_torus")
handled = build_surface_record(handled_points, "torus_with_extra_handle")

/home/gabriel/Desktop/SurgeryTheory/pysurgery/integrations/gudhi_bridge.py:519: UserWarning: Topological Hint: Julia surface triangulation failed (TypeError('Julia: MethodError: no method matching triangulate_surface_delaunay(::PyArray{Float64, 2, true, false, Float64}, ::Float64)\nThe function `triangulate_surface_delaunay` exists, but no method is defined for this combination of argument types.\n\nClosest candidates are:\n  triangulate_surface_delaunay(!Matched::Matrix, ::Float64)\n   @ Main.SurgeryBackend ~/Desktop/SurgeryTheory/pysurgery/bridge/surgery_backend.jl:1154\n  triangulate_surface_delaunay(!Matched::Matrix)\n   @ Main.SurgeryBackend ~/Desktop/SurgeryTheory/pysurgery/bridge/surgery_backend.jl:1154\n')). Falling back to slower pure-Python Delaunay triangulation.
  warnings.warn(
/home/gabriel/Desktop/SurgeryTheory/pysurgery/integrations/gudhi_bridge.py:390: UserWarning: Topological Hint: Point cloud has significant variance in normal direction (6.00e+01). Surface may not be

In [ ]:
def print_surface_summary(rec):
    print(f"\n=== {rec['name']} ===")
    c = rec["counts"]
    print(f"V={c['V']} E={c['E']} F={c['F']} Euler={c['chi']}")
    print(f"H0={rec['homology']['H0']}")
    print(f"H1={rec['homology']['H1']}")
    print(f"H2={rec['homology']['H2']}")
    print(f"H0 generators: rank={rec['generators']['H0'].rank} | {rec['generators']['H0'].message}")
    print(f"H1 generators: rank={rec['generators']['H1'].rank} | {rec['generators']['H1'].message}")
    print(f"H2 generators: rank={rec['generators']['H2'].rank} | {rec['generators']['H2'].message}")


print_surface_summary(clean)
print_surface_summary(handled)

## Plotting Helpers (triangulation + H0/H1/H2 generator overlays)

In [ ]:
def first_h0_vertices(h0_basis, max_points=8):
    verts = []
    for g in h0_basis.generators:
        for s in g.support_simplices:
            if len(s) == 1:
                verts.append(int(s[0]))
    # unique preserving order
    uniq = []
    seen = set()
    for v in verts:
        if v not in seen:
            seen.add(v)
            uniq.append(v)
    return uniq[:max_points]


def edges_from_h1_basis(h1_basis, max_generators=4):
    edge_sets = []
    for g in h1_basis.generators[:max_generators]:
        edge_sets.append([tuple(sorted(e)) for e in g.support_edges])
    return edge_sets


def faces_from_h2_basis(h2_basis, max_faces=200):
    faces = []
    for g in h2_basis.generators:
        for s in g.support_simplices:
            if len(s) == 3:
                faces.append(tuple(s))
    # dedupe and cap
    out = []
    seen = set()
    for f in faces:
        sf = tuple(sorted(f))
        if sf not in seen:
            seen.add(sf)
            out.append(sf)
        if len(out) >= max_faces:
            break
    return out


def add_surface_with_generators(fig, rec, col=1, row=1):
    pts = rec["points"]
    st = rec["st"]
    h0 = rec["generators"]["H0"]
    h1 = rec["generators"]["H1"]
    h2 = rec["generators"]["H2"]

    faces = [tuple(s) for s, _ in st.get_skeleton(2) if len(s) == 3]
    if len(faces) == 0:
        raise ValueError(f"No faces found in simplex tree for {rec['name']}.")

    fi = [f[0] for f in faces]
    fj = [f[1] for f in faces]
    fk = [f[2] for f in faces]

    fig.add_trace(
        go.Mesh3d(
            x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
            i=fi, j=fj, k=fk,
            opacity=0.22,
            color="lightblue",
            name=f"{rec['name']} mesh",
            showlegend=(col == 1),
        ),
        row=row, col=col
    )

    # H0 representatives
    h0_vs = first_h0_vertices(h0)
    if h0_vs:
        fig.add_trace(
            go.Scatter3d(
                x=pts[h0_vs, 0], y=pts[h0_vs, 1], z=pts[h0_vs, 2],
                mode="markers",
                marker=dict(size=6, color="black", symbol="diamond"),
                name=f"{rec['name']} H0 reps",
                showlegend=(col == 1),
            ),
            row=row, col=col
        )

    # H1 generators as colored edge loops
    colors = ["red", "orange", "purple", "green", "magenta", "cyan"]
    for gi, edge_list in enumerate(edges_from_h1_basis(h1, max_generators=4)):
        xs, ys, zs = [], [], []
        for (u, v) in edge_list:
            xs.extend([pts[u, 0], pts[v, 0], None])
            ys.extend([pts[u, 1], pts[v, 1], None])
            zs.extend([pts[u, 2], pts[v, 2], None])

        fig.add_trace(
            go.Scatter3d(
                x=xs, y=ys, z=zs,
                mode="lines",
                line=dict(width=5, color=colors[gi % len(colors)]),
                name=f"{rec['name']} H1 gen {gi}",
                showlegend=(col == 1),
            ),
            row=row, col=col
        )

    # H2 representative faces (first generator support subset)
    h2_faces = faces_from_h2_basis(h2, max_faces=120)
    if h2_faces:
        hi = [f[0] for f in h2_faces]
        hj = [f[1] for f in h2_faces]
        hk = [f[2] for f in h2_faces]
        fig.add_trace(
            go.Mesh3d(
                x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
                i=hi, j=hj, k=hk,
                opacity=0.45,
                color="gold",
                name=f"{rec['name']} H2 support",
                showlegend=(col == 1),
            ),
            row=row, col=col
        )

In [ ]:
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "scene"}, {"type": "scene"}]],
    subplot_titles=(
        "Normal Torus (Reference)",
        "Torus + Extra Handle (Before Surgery)",
    ),
)

add_surface_with_generators(fig, clean, col=1, row=1)
add_surface_with_generators(fig, handled, col=2, row=1)

fig.update_layout(
    title="Triangulated Surfaces with Homology Generator Overlays",
    height=760,
    width=1350,
)
fig.show()

## pySurgery Decision Stage (Before Surgery)

In [ ]:
pre_result = analyze_homeomorphism_2d_result(clean["cc"], handled["cc"])
pre_witness = build_homeomorphism_witness(c1=clean["cc"], c2=handled["cc"], dim=2)

print("Analyzer status:", pre_result.status)
print("Analyzer theorem:", pre_result.theorem)
print("Analyzer reasoning:", pre_result.reasoning)

print("\nWitness status:", pre_witness.status)
if pre_witness.witness is not None:
    print("Witness kind:", pre_witness.witness.kind)
    print("Witness theorem:", pre_witness.witness.theorem)
else:
    print("Missing data:", pre_witness.missing_data)

## Surgery Step: Remove the Extra Handle

For this synthetic demo, we apply a geometric surgery operator that keeps the main torus lobe and re-closes it as a single torus.
Then we re-triangulate and recompute homology.

In [ ]:
def surgery_remove_extra_handle(points, offset=3.0, R=2.2, r=0.75, target_nu=120, target_nv=60, seed=123):
    # Keep points close to left torus lobe (acts like excising extra handle + bridge)
    left_center = np.array([-offset, 0.0, 0.0], dtype=float)
    keep_radius = R + r + 0.35
    keep_mask = np.linalg.norm(points - left_center, axis=1) <= keep_radius
    kept = points[keep_mask].copy()

    # Recenter near origin (surgery gluing in this synthetic pipeline)
    kept[:, 0] += offset

    # Resample a smooth torus scaffold to stabilize triangulation and guarantee closure
    scaffold = sample_torus(
        nu=target_nu, nv=target_nv, R=3.0, r=1.0, center=(0.0, 0.0, 0.0),
        noise=0.003, seed=seed
    )

    # Merge and deduplicate softly
    merged = np.vstack([kept, scaffold])
    merged = np.unique(np.round(merged, 5), axis=0)
    return merged


post_points = surgery_remove_extra_handle(handled_points)
print("post-surgery points:", post_points.shape)

In [ ]:
post = build_surface_record(post_points, "post_surgery_torus")

print_surface_summary(post)

post_result = analyze_homeomorphism_2d_result(clean["cc"], post["cc"])
post_witness = build_homeomorphism_witness(c1=clean["cc"], c2=post["cc"], dim=2)

print("\nPost-surgery analyzer status:", post_result.status)
print("Post-surgery reasoning:", post_result.reasoning)

print("\nPost-surgery witness status:", post_witness.status)
if post_witness.witness is not None:
    print("Post-surgery witness kind:", post_witness.witness.kind)
    print("Post-surgery theorem:", post_witness.witness.theorem)
else:
    print("Post-surgery missing data:", post_witness.missing_data)

In [ ]:
fig2 = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "scene"}, {"type": "scene"}]],
    subplot_titles=(
        "Before Surgery: Torus + Extra Handle",
        "After Surgery: Repaired Torus",
    ),
)

add_surface_with_generators(fig2, handled, col=1, row=1)
add_surface_with_generators(fig2, post, col=2, row=1)

fig2.update_layout(
    title="Surgery Effect on Triangulation and Homology Generators",
    height=760,
    width=1350,
)
fig2.show()

In [ ]:
def short_h(rec):
    return {
        "H0_rank": rec["homology"]["H0"][0],
        "H1_rank": rec["homology"]["H1"][0],
        "H2_rank": rec["homology"]["H2"][0],
        "chi": rec["counts"]["chi"],
    }

print("Clean torus:", short_h(clean))
print("Handled (before surgery):", short_h(handled))
print("Post-surgery:", short_h(post))

print("\nDecision summary:")
print("Before surgery ->", pre_result.status)
print("After surgery  ->", post_result.status)